In [1]:
from pyscf import gto, fci, mcscf
import numpy as np
from gpu4pyscf import scf


/home/mert/.local/lib/python3.10/site-packages/gpu4pyscf/lib/cutensor.py:154: UserWarning: using cupy as the tensor contraction engine.
  warnings.warn(f'using {contract_engine} as the tensor contraction engine.')


In [2]:
# Molecular Information part
# !!!!! in CI part mo not forget the change electrons 
# !!!!! also orbitals are important  

xyz = '''
 C    0.751851   -0.378235    0.000000
 C    0.000000    0.745928    0.000000
 C   -0.751907   -0.378151    0.000000
 H    1.694209   -0.880972    0.000000
 H    0.000140    1.825086    0.000000
 H   -1.694012   -0.881365    0.000000'''
molecule = 'C3H3_rad'
spin = 1
charge =  0


In [9]:
def rhf_calculation(xyz, molecule, spin, charge):
    mol = gto.M(
          atom= xyz,
          basis = 'sto-6g',
          charge = charge,
          verbose = 4 ,
          spin = spin, 
          symmetry = True,
          output  ="./"+molecule+".txt")

    mf = mol.ROHF().run()
    mf.analyze() 
    S_ij = mf.get_ovlp()
    C_ij = mf.mo_coeff
    np.savetxt("C_"+molecule+'mo_coeff.txt', C_ij,fmt = '%.8f')
    np.savetxt('S_'+molecule+'_overlap.txt', S_ij,fmt = '%.8f')

    return mf, mol 

In [11]:
def orb_occ_table(orb_energy,orb_occ):
    print ("orbital  energy   occ")

    for i in range (0,(len(orb_occ))):
        print(i ,"   ", orb_energy[i],"  ",orb_occ[i])
              
    for i in range(len(orb_occ) - 1):
        if orb_occ[i] >=1   and orb_occ[i+1] == 0:
            homo = i
            lumo = i + 1
            print(f"Homo_index: {homo}, Lumo_index: {lumo}")
    return homo, lumo


In [12]:
mf, mol  = rhf_calculation( xyz , molecule, spin, charge )
homo, lumo = orb_occ_table(mf.mo_energy, mf.mo_occ)

overwrite output file: ./C3H3_rad.txt
orbital  energy   occ
0     -11.229281150290017    2.0
1     -11.137320751666138    2.0
2     -11.137160721097978    2.0
3     -1.1048745025705102    2.0
4     -0.7348475740726621    2.0
5     -0.7293857102005641    2.0
6     -0.6394283419856142    2.0
7     -0.4655805816796541    2.0
8     -0.4187901835095279    2.0
9     -0.3738869706626873    2.0
10     0.09871384942332398    1.0
11     0.3092861954308196    0.0
12     0.5251359483678482    0.0
13     0.6131889460854095    0.0
14     0.6250328214431478    0.0
15     0.65651132205086    0.0
16     1.035862373427055    0.0
17     1.1447138370251055    0.0
Homo_index: 10, Lumo_index: 11


In [13]:
n_elec= 3
n_orb = 3 # number of orbital can be determined based on the number of pi  orbitals 
casscf = mcscf.CASSCF(mf, n_orb, n_elec)
'''output in order
e_tot
e_cas
Fci vecs 
mo_coeff
mo_energy'''
mo = casscf.sort_mo([9,11,12])# the numbers are from the index of pi orbitals 
casscf.kernel(mo) 


(np.float64(-114.89229675722594),
 np.float64(-2.2500393222106396),
 FCIvector([[-9.52136600e-01, -2.25075615e-05, -1.33566373e-01],
            [-2.18032736e-04, -2.04166163e-01, -3.84180715e-05],
            [-7.06174001e-02,  2.74669422e-05, -1.70074189e-01]]),
 NPArrayWithTag([[ 7.22497112e-03,  5.74344670e-01,  8.12913443e-01,
                  -1.33721799e-01,  1.37718977e-01,  7.84118839e-02,
                  -3.65820745e-02, -2.49008578e-02, -2.00673751e-02,
                   0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
                  -1.25665186e-01,  9.98028140e-02,  1.09069430e-01,
                   6.53063150e-02,  4.39739979e-02, -3.32483556e-02],
                 [-4.32252441e-03,  1.02215043e-02,  1.97887347e-02,
                   3.15066420e-01, -4.40689804e-01, -2.46727764e-01,
                   1.64171864e-01,  1.15998252e-01,  1.26372418e-01,
                   0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
                   7.66904236e-01, -6.737259

In [14]:
casscf.fcisolver.max_cycle = 3000
casscf.canonicalize(mo_coeff=None, ci=None, eris=None, sort=False, cas_natorb=False, casdm1=None, verbose=3, with_meta_lowdin=False);

filename = f"{molecule}_Ci_coeffs.txt"

with open(filename, 'w', encoding='utf-8') as f:
    
    f.write("det-alpha,     det-beta,    CI coefficients\n")
    
    
    for c, ia, ib in casscf.fcisolver.large_ci(casscf.ci, n_orb, n_elec, tol=.000000001, return_strs=False):
        # Satır formatını oluşturalım
        line = f" {str(ia):<12} {str(ib):<12} {c:.15f}\n"
        
        #write in file 
        f.write(line)
        #show in terminal 
        print(line.strip())

[0 1]        [0]          -0.952136599670907
[0 1]        [1]          -0.000022507561459
[0 1]        [2]          -0.133566373272864
[0 2]        [0]          -0.000218032735618
[0 2]        [1]          -0.204166163040271
[0 2]        [2]          -0.000038418071489
[1 2]        [0]          -0.070617400091369
[1 2]        [1]          0.000027466942168
[1 2]        [2]          -0.170074189389107
